In [1]:
import requests
from urllib.parse import urljoin
from bs4 import BeautifulSoup

URL = "https://www.niftyindices.com/indices/equity/sectoral-indices/nifty-metal"

response = requests.get(URL, headers={"User-Agent": "Mozilla/5.0"})
soup = BeautifulSoup(response.text, "html.parser")

link = soup.find("a", string="Index Constituent")
relative_href = link["href"]

absolute_url = urljoin(URL, relative_href)

file_response = requests.get(absolute_url, headers={"User-Agent": "Mozilla/5.0"})

with open("ind_niftymetallist.csv", "wb") as f:
    f.write(file_response.content)

print("Downloaded successfully.")

Downloaded successfully.


In [2]:
import requests
from bs4 import BeautifulSoup


def get_csv_download_link(page_url):
    session = requests.Session()

    headers = {
        "User-Agent": "Mozilla/5.0",
        "Accept-Language": "en-US,en;q=0.9",
    }

    session.headers.update(headers)

    # Important: First visit homepage (NSE bot protection)
    session.get("https://www.nseindia.com")

    response = session.get(page_url)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "lxml")

    # Find anchor tag containing .csv in href
    link = soup.find("a", href=lambda href: href and href.endswith(".csv"))

    if not link:
        raise ValueError("CSV link not found")

    return link["href"], session


def download_csv(page_url, save_as):
    file_url, session = get_csv_download_link(page_url)

    file_response = session.get(file_url)
    file_response.raise_for_status()

    with open(save_as, "wb") as f:
        f.write(file_response.content)

    print(f"Downloaded: {save_as}")


# Example usage
download_csv(
    "https://www.nseindia.com/static/products-services/indices-niftysmallcap250-index",
    "ind_niftysmallcap250list.csv",
)

Downloaded: ind_niftysmallcap250list.csv
